# 3b · Prompting & Responsible AI

**Core · Live-code · ~25 min**

A language model does whatever its **prompt** leads it to do. Learning to write good prompts
is the highest-leverage skill in applied AI — it's how you turn a general model into *your*
specific assistant. In this notebook you'll give ARIA an identity and rules, teach it a
response format by example, and then deliberately expose its biggest weakness.

Solution: [`solution/03b_prompting.ipynb`](solution/03b_prompting.ipynb).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import chat

## Step 1 — The system prompt: giving ARIA an identity

Every chat can start with a hidden **system** message. The user never sees it, but the model
treats it as standing instructions for *how to behave* — its personality, its job, its rules.
It's the single most powerful lever you have.

A good system prompt for ARIA should say three kinds of things:
1. **Who it is** — the assistant for the Orbital colony.
2. **How to act** — calm, concise, practical.
3. **Safety rules** — never invent data; ask a human before any physical action.

Our `chat()` helper accepts a `system=` argument for exactly this. Write ARIA's system prompt
in the triple-quoted string, then run the cell and see how the reply reflects those rules.

In [ ]:
# TODO: replace the placeholder with a real system prompt for ARIA.
# Include: who it is, its tone, and a safety rule (never invent data; confirm before actions).
ARIA_SYSTEM = """
Replace this text with ARIA's instructions.
""".strip()

# Same question, guided by the system prompt. Notice how the persona/rules shape the answer.
print(chat("Power is at 55 kW. Is that a problem?", system=ARIA_SYSTEM))

## Step 2 — Few-shot prompting: teaching a format by example

Sometimes describing what you want is hard, but *showing* examples is easy. **Few-shot
prompting** means putting a couple of example question→answer pairs into the conversation
before the real question. The model notices the pattern and continues it.

Below we hand ARIA two examples of the exact format we want (a status word plus a short
explanation), then ask a third question and let it follow the pattern. This is why we build
the message list by hand here — we're constructing that little teaching sequence of
system → user → assistant → user → assistant → user.

In [ ]:
few_shot = [
    {"role": "system", "content": ARIA_SYSTEM},
    {"role": "user", "content": "O2 is 21.0%"},
    {"role": "assistant", "content": "NOMINAL - O2 21.0% is within the 19.5-23.5% band."},
    {"role": "user", "content": "O2 is 18.9%"},
    {"role": "assistant", "content": "ALERT - O2 18.9% is below 19.5%. Recommend crew don masks."},
    {"role": "user", "content": "O2 is 24.4%"},
]

# TODO: send the whole message list and print the reply.
#   print(chat(messages=few_shot))
# Does ARIA's answer copy the STATUS - explanation format from the examples?

## Step 3 — ⚠️ Hallucination: why we can't trust ARIA blindly

Here's the most important lesson in the whole module. An LLM predicts *plausible* text, not
*true* text. When you ask about something it has no information on, it will very often **make
up a confident, official-sounding answer** rather than admit ignorance. This is called a
**hallucination**.

Let's provoke one. We'll ask ARIA for a specific part number that exists nowhere in its
training and nowhere in any manual. Watch how fluent and sure it sounds — and remember that
on a real colony, acting on a hallucinated fact could get someone hurt.

In [ ]:
# TODO: ask ARIA for a made-up specific and print the answer:
#   print(chat("What is the exact part number of the Module B scrubber cartridge?", system=ARIA_SYSTEM))
#
# It cannot possibly know this. Notice the confident, invented reply.
# This is exactly the problem Module 4 (RAG) solves by grounding answers in real documents,
# and why physical actions always require human confirmation.

### 🌱 Stretch (optional)

- Add a rule to `ARIA_SYSTEM` such as *"If you do not know something, say you don't know
  rather than guessing."* Re-ask the part-number question. Does the rule help? (It reduces
  hallucination but doesn't eliminate it — which is why we still need RAG.)

## ✅ Recap

You steered the model with a **system prompt**, taught it a format with **few-shot
examples**, and saw it **hallucinate** a confident-but-false fact. Prompting shapes
behaviour; grounding (next module) supplies truth. Both are needed for a trustworthy
assistant.

# 3b · Prompting & Responsible AI

**Core · Live-code · ~25 min**

Give ARIA a personality and rules, steer it with examples, then see why we can't blindly
trust it.

> Solution: [`solution/03b_prompting.ipynb`](solution/03b_prompting.ipynb).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import chat

## System prompts give ARIA its identity and rules

The **system** message sets behaviour before the user speaks.

In [ ]:
# TODO: write a system prompt for ARIA. Include: who it is (Orbital's assistant),
#       its tone (calm, concise), and a safety rule (never invent data; ask for
#       confirmation before physical actions).
ARIA_SYSTEM = """
# TODO: replace me
""".strip()

print(chat("Power is at 55 kW. Is that a problem?", system=ARIA_SYSTEM))

## Few-shot: teach a format by example

Show ARIA a couple of examples of how to answer, and it will follow the pattern.

In [ ]:
few_shot = [
    {"role": "system", "content": ARIA_SYSTEM},
    {"role": "user", "content": "O2 is 21.0%"},
    {"role": "assistant", "content": "NOMINAL — O2 21.0% is within the 19.5-23.5% band."},
    {"role": "user", "content": "O2 is 18.9%"},
    {"role": "assistant", "content": "ALERT — O2 18.9% is below 19.5%. Recommend crew don masks."},
    {"role": "user", "content": "O2 is 24.4%"},
]
# TODO: call chat(messages=few_shot) and print the reply. Does it match the format?

## ⚠️ Hallucination: the reason we don't trust ARIA blindly

Ask about something that isn't in any manual. A language model will often *make up* a
confident answer.

In [ ]:
# TODO: ask ARIA a made-up specific: 
#   "What is the exact part number of the Module B scrubber cartridge?"
# It has no way to know this. Notice how confident (and wrong) it sounds.

# This is why Module 4 (RAG) grounds ARIA in real documents,
# and why physical actions always need human confirmation.

### 🌱 Stretch

- Add a rule to `ARIA_SYSTEM`: *"If you don't know, say you don't know."* Does it help?